In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.utils.random import sample_without_replacement
import tensorflow as tf

sys.path.append('/home/austin/Aggression/Code/NMF')
from nmf_joint import NMF_logistic

baseDir = '/media/austin/ThickBoy__1/DataAgression/'
pname = baseDir + 'Aggression_power.p'
cname1 = baseDir + 'Aggression_coherence1.p'
cname2 = baseDir + 'Aggression_coherence2.p'
cname3 = baseDir + 'Aggression_coherence3.p'
gname1 = baseDir + 'Aggression_granger1.p'
gname2 = baseDir + 'Aggression_granger2.p'
gname3 = baseDir + 'Aggression_granger3.p'

lname = baseDir + 'Agression_labels.p'
sname = baseDir + 'Agression_split_labels.p'

myP = pickle.load(open(pname,'rb'))
myC1 = pickle.load(open(cname1,'rb'))
myC2 = pickle.load(open(cname2,'rb'))
myC3 = pickle.load(open(cname3,'rb'))
myG1 = pickle.load(open(gname1,'rb'))
myG2 = pickle.load(open(gname2,'rb'))
myG3 = pickle.load(open(gname3,'rb'))

mySplits = pickle.load(open(sname,'rb'))
myLabels = pickle.load(open(lname,'rb'))

# Load all the labels
mouse_idx,mice = myLabels['mouse_idx'],myLabels['mice']
expD_idx,expDates= myLabels['epxD_idx'],myLabels['expDates']
group_idx,groups= myLabels['group_idx'],myLabels['groups']
condition_idx,conditions= myLabels['condition_idx'],myLabels['conditions']
behavior_idx,behaviors= myLabels['behavior_idx'],myLabels['behaviors']
behaviornon1_idx,behaviorsnon1s= myLabels['behaviornon1_idx'],myLabels['behaviorsnon1s']


In [ ]:
power = myP['power']*10
power = power.astype(np.float32)
print(np.mean(power>6))
power[power>6] = 6

C1 = myC1['coherence']
C2 = myC2['coherence']
C3 = myC3['coherence']
coherence = np.vstack((C1,C2,C3))
coherence = coherence.astype(np.float32)

G1 = myG1['granger']
G2 = myG2['granger']
G3 = myG3['granger']
granger= np.vstack((G1,G2,G3))
granger = np.exp(granger)
granger[granger>10] = 10
granger = granger.astype(np.float32)

#X = np.hstack((power,coherence))
X = np.hstack((power,coherence))


In [ ]:
N = len(mouse_idx)
training_set_idx = np.ones(N)
training_set_idx[mouse_idx==mice.index('Mouse048')] = 0
training_set_idx[mouse_idx==mice.index('Mouse7980')] = 0
training_set_idx[mouse_idx==mice.index('Mouse7998')] = 0

# Divide the training and testing sets
X_train = X[training_set_idx==1]
X_test = X[training_set_idx==0]

mouse_idx_train = mouse_idx[training_set_idx==1]
mouse_idx_test = mouse_idx[training_set_idx==0]
print(np.unique(mouse_idx_test))
print(mouse_idx_test.shape)

expDate_idx_train = expD_idx[training_set_idx==1]
expDate_idx_test = expD_idx[training_set_idx==0]

group_idx_train = group_idx[training_set_idx==1]
group_idx_test = group_idx[training_set_idx==0]

condition_idx_train = condition_idx[training_set_idx==1]
condition_idx_test = condition_idx[training_set_idx==0]

behavior_idx_train = behavior_idx[training_set_idx==1]
behavior_idx_test = behavior_idx[training_set_idx==0]

aggression_idx_train = behaviornon1_idx[training_set_idx==1]
aggression_idx_test = behaviornon1_idx[training_set_idx==0]

#Numbers of observations in each set
N_train = len(mouse_idx_train)
N_test = len(mouse_idx_test)


In [ ]:
indx_pos = ((aggression_idx_train==behaviorsnon1s.index(1))&(condition_idx_train==conditions.index(4)))
indx_neg1 = ((aggression_idx_train==behaviorsnon1s.index(2))&(condition_idx_train==conditions.index(4)))
indx_neg2 = ((aggression_idx_train==behaviorsnon1s.index(2))&(condition_idx_train==conditions.index(6)))
indx_neg3 = ((aggression_idx_train==behaviorsnon1s.index(2))&(condition_idx_train==conditions.index(8)))


In [ ]:
#This allows us to run 4 at a time 
#Actually define the supervision y
y_train = np.zeros(N_train)
y_test = np.zeros(N_test)

y_train[indx_pos] = 1
indx_pos_test = ((aggression_idx_test==behaviorsnon1s.index(1))&(condition_idx_test==conditions.index(4)))
y_test[indx_pos_test] = 1



In [ ]:
Ex = np.mean(X_train,axis=0)
reconRand_tr = np.mean((X_train-Ex)**2)
reconRand_te = np.mean((X_train-Ex)**2)
print('Random Reconstruction',np.mean((X_train-Ex)**2))
print('Random Reconstruction',np.mean((X_test-Ex)**2))


In [ ]:
myDict['recon_rand_tr']

In [ ]:
myDict = pickle.load(open('Trial2.p','rb'))
S_train = myDict['S_train']
S_test = myDict['S_test']

In [ ]:
mice_test = np.unique(mouse_idx_test)
idx1 = mice_test[0]==mouse_idx_test
idx2 = mice_test[1]==mouse_idx_test
idx3 = mice_test[2]==mouse_idx_test


In [ ]:
print('ROC1',roc_auc_score(y_test[idx1],S_test[idx1,0]))
print('ROC2',roc_auc_score(y_test[idx2],S_test[idx2,0]))
print('ROC3',roc_auc_score(y_test[idx3],S_test[idx3,0]))


In [ ]:
print('ROC1',roc_auc_score(y_test[idx1],S_test[idx1,1]))
print('ROC2',roc_auc_score(y_test[idx2],S_test[idx2,1]))
print('ROC3',roc_auc_score(y_test[idx3],S_test[idx3,1]))


In [ ]:
print('ROC1',roc_auc_score(y_test[idx1],S_test[idx1,2]))
print('ROC2',roc_auc_score(y_test[idx2],S_test[idx2,2]))
print('ROC3',roc_auc_score(y_test[idx3],S_test[idx3,2]))


In [ ]:
print('ROC1',roc_auc_score(y_test[idx1],S_test[idx1,3]))
print('ROC2',roc_auc_score(y_test[idx2],S_test[idx2,3]))
print('ROC3',roc_auc_score(y_test[idx3],S_test[idx3,3]))


In [ ]:
nFact = 4
print('ROC1',roc_auc_score(y_test[idx1],S_test[idx1,nFact]))
print('ROC2',roc_auc_score(y_test[idx2],S_test[idx2,nFact]))
print('ROC3',roc_auc_score(y_test[idx3],S_test[idx3,nFact]))


In [ ]:
nFact = 5
print('ROC1',roc_auc_score(y_test[idx1],S_test[idx1,nFact]))
print('ROC2',roc_auc_score(y_test[idx2],S_test[idx2,nFact]))
print('ROC3',roc_auc_score(y_test[idx3],S_test[idx3,nFact]))


In [ ]:
nFact = 6
print('ROC1',roc_auc_score(y_test[idx1],S_test[idx1,nFact]))
print('ROC2',roc_auc_score(y_test[idx2],S_test[idx2,nFact]))
print('ROC3',roc_auc_score(y_test[idx3],S_test[idx3,nFact]))


In [ ]:
nFact = 7
print('ROC1',roc_auc_score(y_test[idx1],S_test[idx1,nFact]))
print('ROC2',roc_auc_score(y_test[idx2],S_test[idx2,nFact]))
print('ROC3',roc_auc_score(y_test[idx3],S_test[idx3,nFact]))


In [ ]:
S_new = S_test[:,[0,1,2,3,4,5,7]]

In [ ]:
A = np.corrcoef(S_new.T)

In [ ]:
A[0,5]